# NLP Fundamentals
### Interactive Notebook for AI/ML Interview Preparation

📺 **Video Lecture:** [https://youtu.be/956n9PUi-zg](https://youtu.be/956n9PUi-zg)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
import re

import spacy
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score

# Download required NLTK data
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)

# Load spaCy model
try:
    nlp = spacy.load('en_core_web_sm')
except:
    print('Please run: python -m spacy download en_core_web_sm')

np.random.seed(42)
print('All libraries loaded successfully!')

---
## 1. Text Preprocessing Pipeline

**Objective:** Clean and normalize text using both regex and spaCy approaches.

In [ ]:
sample_texts = [
    'Machine Learning is amazing! It powers many AI applications.',
    'Deep Learning, a subset of ML, uses neural networks.',
    'Natural Language Processing (NLP) helps machines understand text.',
    'AI and Machine Learning are transforming industries worldwide!',
]

# Method 1: Regex-based preprocessing
def preprocess_regex(text):
    """Clean text using regex"""
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)  # remove punctuation and digits
    text = re.sub(r'\s+', ' ', text)  # remove extra whitespace
    return text.strip()

# Method 2: spaCy-based preprocessing
def preprocess_spacy(text):
    """Clean text using spaCy"""
    doc = nlp(text.lower())
    # Remove punctuation and special characters
    tokens = [token.text for token in doc if not token.is_punct and token.is_alpha]
    return ' '.join(tokens)

print('\n=== TEXT PREPROCESSING COMPARISON ===\n')
for text in sample_texts:
    print(f'Original:\n  {text}')
    print(f'Regex method:\n  {preprocess_regex(text)}')
    print(f'spaCy method:\n  {preprocess_spacy(text)}')
    print()

---
## 2. Tokenization

**Objective:** Compare whitespace splitting, NLTK tokenizer, and spaCy tokenizer.

In [ ]:
test_sentence = "Dr. Smith went to New York City. He said, 'It's amazing!' Don't you agree?"

print('\n=== TOKENIZATION COMPARISON ===\n')
print(f'Original: {test_sentence}\n')

# Method 1: Simple whitespace split
whitespace_tokens = test_sentence.split()
print(f'Whitespace split ({len(whitespace_tokens)} tokens):')
print(f'  {whitespace_tokens}\n')

# Method 2: NLTK tokenizer
nltk_tokens = word_tokenize(test_sentence)
print(f'NLTK word_tokenize ({len(nltk_tokens)} tokens):')
print(f'  {nltk_tokens}\n')

# Method 3: spaCy tokenizer
doc = nlp(test_sentence)
spacy_tokens = [token.text for token in doc]
print(f'spaCy tokenizer ({len(spacy_tokens)} tokens):')
print(f'  {spacy_tokens}\n')

# Comparison table
print('Summary:')
print(f'  Whitespace:  {len(whitespace_tokens)} tokens (loses punctuation info)')
print(f'  NLTK:        {len(nltk_tokens)} tokens (separates punctuation)')
print(f'  spaCy:       {len(spacy_tokens)} tokens (separates punctuation, advanced)')

---
## 3. Stopword Removal

**Objective:** Remove common words and analyze the impact on vocabulary.

In [ ]:
# Get NLTK stopwords
stop_words = set(stopwords.words('english'))

corpus = [
    'The machine learning model is very accurate.',
    'It can process large amounts of data quickly.',
    'Natural language processing is a key NLP technique.',
    'Deep learning models require a lot of computational power.',
]

print('\n=== STOPWORD REMOVAL IMPACT ===\n')

for i, text in enumerate(corpus, 1):
    tokens = word_tokenize(text.lower())
    tokens_clean = [t for t in tokens if t.isalpha()]
    
    with_stops = [t for t in tokens_clean if t not in stop_words]
    without_stops = [t for t in tokens_clean if t in stop_words]
    
    print(f'Doc {i}:')
    print(f'  Original tokens ({len(tokens_clean)}): {tokens_clean}')
    print(f'  After stopword removal ({len(with_stops)}): {with_stops}')
    print(f'  Stopwords removed: {without_stops}\n')

# Aggregate statistics
print('Overall Impact:')
all_tokens = []
all_tokens_no_stops = []
for text in corpus:
    tokens = [t.lower() for t in word_tokenize(text) if t.isalpha()]
    all_tokens.extend(tokens)
    all_tokens_no_stops.extend([t for t in tokens if t not in stop_words])

print(f'  Total tokens: {len(all_tokens)}')
print(f'  After stopword removal: {len(all_tokens_no_stops)}')
print(f'  Reduction: {len(all_tokens) - len(all_tokens_no_stops)} tokens ({100*(len(all_tokens) - len(all_tokens_no_stops))/len(all_tokens):.1f}%)')
print(f'  Unique vocabulary (with stops): {len(set(all_tokens))}')
print(f'  Unique vocabulary (without stops): {len(set(all_tokens_no_stops))}')

---
## 4. Stemming vs Lemmatization

**Objective:** Compare NLTK stemming with spaCy lemmatization.

In [ ]:
stemmer = PorterStemmer()

test_sentences = [
    'Running and runners are important for athletics.',
    'The studies studied by students involved multiple universities.',
    'Historically, history books have documented historical events.',
]

print('\n=== STEMMING VS LEMMATIZATION ===\n')

for sentence in test_sentences:
    print(f'Original: {sentence}\n')
    
    # NLTK Stemming
    tokens = word_tokenize(sentence.lower())
    stems = [stemmer.stem(t) for t in tokens if t.isalpha()]
    print(f'NLTK Stemming:  {stems}')
    
    # spaCy Lemmatization
    doc = nlp(sentence)
    lemmas = [token.lemma_ for token in doc if token.is_alpha]
    print(f'spaCy Lemmatization: {lemmas}')
    print()

# Comparison table
print('\n=== DETAILED COMPARISON TABLE ===\n')
print('Word        | Stem (Porter) | Lemma (spaCy)')
print('-' * 45)

test_words = ['running', 'studies', 'history', 'processing', 'played', 'better']
for word in test_words:
    stem = stemmer.stem(word)
    token = nlp(word)[0]
    lemma = token.lemma_
    print(f'{word:11} | {stem:13} | {lemma}')

---
## 5. Part-of-Speech (POS) Tagging

**Objective:** Analyze POS tags using spaCy and visualize tag distribution.

In [ ]:
pos_text = '''
Apple is looking at buying British startup for 1 billion dollars.
The quick brown fox jumps over the lazy dog.
'''

doc = nlp(pos_text)

print('\n=== POS TAGGING WITH SPACY ===\n')
print('Token          | POS  | Description')
print('-' * 50)

pos_counter = Counter()
for token in doc:
    if not token.is_punct and not token.is_space:
        print(f'{token.text:14} | {token.pos_:4} | {spacy.explain(token.pos_)}')
        pos_counter[token.pos_] += 1

print(f'\nPOS Tag Distribution:')
for pos_tag, count in pos_counter.most_common():
    print(f'  {pos_tag}: {count}')

---
## 6. Named Entity Recognition (NER)

**Objective:** Extract and display named entities with their labels.

In [ ]:
ner_texts = [
    'Apple Inc. was founded by Steve Jobs in Cupertino, California.',
    'Barack Obama was the 44th President of the United States.',
    'Tesla Inc. manufactures electric vehicles and is headquartered in Palo Alto.',
    'On January 15, 2023, ChatGPT reached 100 million users.',
]

print('\n=== NAMED ENTITY RECOGNITION ===\n')

for text in ner_texts:
    doc = nlp(text)
    print(f'Text: {text}')
    print('Entities:')
    if doc.ents:
        for ent in doc.ents:
            print(f'  - {ent.text:20} | {ent.label_:10} | {spacy.explain(ent.label_)}')
    else:
        print('  (No entities found)')
    print()

# Aggregate entity statistics
all_docs = [nlp(text) for text in ner_texts]
all_entities = [ent for doc in all_docs for ent in doc.ents]
entity_label_counter = Counter([ent.label_ for ent in all_entities])

print('\nEntity Label Distribution (across all texts):')
for label, count in entity_label_counter.most_common():
    print(f'  {label}: {count}')

---
## 7. Dependency Parsing

**Objective:** Show syntactic relationships using dependency parsing.

In [ ]:
dep_sentence = 'The quick brown fox jumps over the lazy dog.'
doc = nlp(dep_sentence)

print('\n=== DEPENDENCY PARSING ===\n')
print(f'Sentence: {dep_sentence}\n')
print('Token          | POS  | Head       | Dep Type')
print('-' * 55)

for token in doc:
    if not token.is_punct:
        print(f'{token.text:14} | {token.pos_:4} | {token.head.text:10} | {token.dep_}')

print(f'\n\nDependency Tree (head-child relationships):')
for token in doc:
    if token.head != token:  # Skip root
        print(f'  {token.head.text} --{token.dep_}-> {token.text}')

# Additional example: Find all verb-subject relationships
print(f'\n\nSubject-Verb Relationships:')
for token in doc:
    if token.dep_ == 'nsubj':  # nominal subject
        print(f'  Subject: {token.text} | Verb: {token.head.text}')

---
## 8. Bag of Words & TF-IDF

**Objective:** Use sklearn's CountVectorizer and TfidfVectorizer on a corpus.

In [ ]:
bow_corpus = [
    'machine learning models require training data',
    'deep learning uses neural networks',
    'natural language processing processes text',
    'machine learning and deep learning are subsets of artificial intelligence',
    'neural networks process data through layers',
]

print('\n=== BAG OF WORDS (CountVectorizer) ===\n')

# CountVectorizer for Bag of Words
cv = CountVectorizer(lowercase=True, stop_words='english')
bow_matrix = cv.fit_transform(bow_corpus).toarray()
vocab = cv.get_feature_names_out()

print(f'Vocabulary size: {len(vocab)}')
print(f'BoW matrix shape: {bow_matrix.shape} (documents x vocabulary)')
print(f'Vocabulary: {list(vocab)}')
print(f'\nBoW Matrix:')
for i, doc in enumerate(bow_corpus):
    print(f'  Doc {i+1}: {doc}')
    print(f'    Counts: {bow_matrix[i]} (sum={bow_matrix[i].sum()})')

print(f'\n\n=== TF-IDF (TfidfVectorizer) ===\n')

# TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer(lowercase=True, stop_words='english')
tfidf_matrix = tfidf_vectorizer.fit_transform(bow_corpus).toarray()

print(f'TF-IDF matrix shape: {tfidf_matrix.shape}')
print(f'\nTop TF-IDF words per document:')
for i, doc in enumerate(bow_corpus):
    top_indices = np.argsort(tfidf_matrix[i])[-3:][::-1]
    top_words = [(vocab[idx], tfidf_matrix[i, idx]) for idx in top_indices]
    print(f'  Doc {i+1}: {[(w, f"{v:.3f}") for w, v in top_words]}')

---
## 9. Text Classification

**Objective:** Build a simple sentiment classifier using TF-IDF + Naive Bayes.

In [ ]:
# Create a small sentiment dataset
sentiment_corpus = [
    # Positive reviews
    ('This product is absolutely amazing and exceeded my expectations!', 1),
    ('I love this! Best purchase ever, highly recommend.', 1),
    ('Fantastic quality and fast shipping. Very satisfied.', 1),
    ('Great customer service and excellent product quality.', 1),
    ('Outstanding value for money. Will buy again!', 1),
    ('Simply wonderful! Everyone should have this product.', 1),
    ('Perfect! Exactly what I needed. Very happy.', 1),
    ('Exceptional quality. Far exceeded my expectations.', 1),
    ('Fantastic experience from start to finish.', 1),
    ('Incredible product. Cannot recommend enough!', 1),
    ('This is the best thing I have ever bought.', 1),
    ('Amazing quality and super fast delivery service.', 1),
    ('Excellent product at great price. Very pleased.', 1),
    ('I am completely satisfied with my purchase here.', 1),
    ('Wonderful quality. Absolutely delighted with it.', 1),
    # Negative reviews
    ('This product is terrible and a complete waste of money.', 0),
    ('Worst purchase ever. Totally disappointed.', 0),
    ('Poor quality and awful customer service.', 0),
    ('Horrible product. Do not recommend at all.', 0),
    ('Defective item. Very frustrated and upset.', 0),
    ('Completely useless. Returning immediately.', 0),
    ('Terrible experience. Worst company ever.', 0),
    ('Awful quality and very disappointing results.', 0),
    ('This is garbage. Waste of time and money.', 0),
    ('Extremely poor quality. Very unsatisfied.', 0),
    ('Horrible product that broke immediately.', 0),
    ('Bad experience. Would not buy again.', 0),
    ('Defective and useless product.', 0),
    ('Terrible quality. Total disappointment.', 0),
    ('Worst item ever. Absolutely terrible.', 0),
]

texts = [t[0] for t in sentiment_corpus]
labels = [t[1] for t in sentiment_corpus]

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.3, random_state=42
)

# TF-IDF vectorization
tfidf_vec = TfidfVectorizer(lowercase=True, stop_words='english', max_features=100)
X_train_tfidf = tfidf_vec.fit_transform(X_train)
X_test_tfidf = tfidf_vec.transform(X_test)

# Train Naive Bayes classifier
clf = MultinomialNB()
clf.fit(X_train_tfidf, y_train)

# Predictions and evaluation
y_pred = clf.predict(X_test_tfidf)

print('\n=== SENTIMENT CLASSIFICATION RESULTS ===\n')
print(f'Training set size: {len(X_train)}')
print(f'Test set size: {len(X_test)}')
print(f'\nModel: TF-IDF + Multinomial Naive Bayes')
print(f'\nPerformance Metrics:')
print(f'  Accuracy:  {accuracy_score(y_test, y_pred):.3f}')
print(f'  Precision: {precision_score(y_test, y_pred):.3f}')
print(f'  Recall:    {recall_score(y_test, y_pred):.3f}')

print(f'\nSample Predictions:')
for i in range(min(5, len(X_test))):
    true_label = 'Positive' if y_test.iloc[i] == 1 else 'Negative'
    pred_label = 'Positive' if y_pred[i] == 1 else 'Negative'
    match = '✓' if y_test.iloc[i] == y_pred[i] else '✗'
    print(f'  {match} "{X_test.iloc[i][:50]}..."')
    print(f'    True: {true_label} | Predicted: {pred_label}\n')

---
## 10. N-grams

**Objective:** Extract and analyze bigrams and trigrams.

In [ ]:
ngram_corpus = [
    'machine learning is a subset of artificial intelligence',
    'deep learning uses neural networks for processing',
    'natural language processing helps machines understand text',
    'machine learning and deep learning are transforming the world',
]

print('\n=== N-GRAMS ANALYSIS ===\n')

# Bigrams using sklearn
print('--- BIGRAMS ---')
bigram_vectorizer = CountVectorizer(ngram_range=(2, 2), lowercase=True, stop_words='english')
bigram_matrix = bigram_vectorizer.fit_transform(ngram_corpus).toarray()
bigrams = bigram_vectorizer.get_feature_names_out()

bigram_counts = bigram_matrix.sum(axis=0)
top_bigrams = np.argsort(bigram_counts)[-5:][::-1]

print(f'Total unique bigrams: {len(bigrams)}')
print(f'\nTop 5 bigrams:')
for idx in top_bigrams:
    print(f'  - {bigrams[idx]}: {int(bigram_counts[idx])} occurrences')

# Trigrams using sklearn
print(f'\n--- TRIGRAMS ---')
trigram_vectorizer = CountVectorizer(ngram_range=(3, 3), lowercase=True, stop_words='english')
trigram_matrix = trigram_vectorizer.fit_transform(ngram_corpus).toarray()
trigrams = trigram_vectorizer.get_feature_names_out()

trigram_counts = trigram_matrix.sum(axis=0)
top_trigrams = np.argsort(trigram_counts)[-5:][::-1]

print(f'Total unique trigrams: {len(trigrams)}')
print(f'\nAll trigrams (with counts):')
for idx in np.argsort(trigram_counts):
    if trigram_counts[idx] > 0:
        print(f'  - {trigrams[idx]}: {int(trigram_counts[idx])} occurrences')

# Manual bigram extraction using spaCy
print(f'\n--- MANUAL BIGRAM EXTRACTION (spaCy) ---')
for text in ngram_corpus:
    doc = nlp(text)
    bigrams_manual = [f'{doc[i].text} {doc[i+1].text}' for i in range(len(doc)-1) if not doc[i].is_stop and not doc[i+1].is_stop]
    print(f'Text: {text}')
    print(f'  Bigrams (no stops): {bigrams_manual}\n')

---
## Interview Takeaways

### Key NLP Concepts:

1. **Text Preprocessing**
   - Lowercasing, punctuation removal, whitespace normalization
   - Both regex and spaCy are viable; spaCy is more flexible

2. **Tokenization**
   - Whitespace split: simple but naive
   - NLTK: separates punctuation, handles contractions
   - spaCy: advanced, handles complex cases, faster

3. **Stopword Removal**
   - Removes common words (the, is, a, etc.) to reduce noise
   - Can improve downstream tasks but may lose information

4. **Stemming vs Lemmatization**
   - Stemming (Porter): aggressive, rule-based, may produce non-words
   - Lemmatization: morphological, returns valid dictionary words
   - Lemmatization typically preferred for NLP tasks

5. **POS Tagging**
   - Identifies grammatical role of each token (NOUN, VERB, ADJ, etc.)
   - Essential for syntax-aware NLP tasks

6. **Named Entity Recognition (NER)**
   - Extracts entities: PERSON, ORG, GPE, DATE, etc.
   - Critical for information extraction and knowledge graphs

7. **Dependency Parsing**
   - Models syntactic relationships (subject-verb, object, etc.)
   - Foundation for semantic understanding

8. **Bag of Words (BoW)**
   - Simple word counting; ignores word order
   - Fast, interpretable, baseline for many tasks

9. **TF-IDF**
   - Term Frequency - Inverse Document Frequency
   - Weights words by importance; upweights rare, domain-specific words
   - Better than raw BoW for document similarity and classification

10. **Text Classification**
    - Common pipeline: TF-IDF → Classifier (Naive Bayes, SVM, Logistic Regression)
    - Sentiment analysis, spam detection, topic classification

11. **N-grams**
    - Bigrams (2-word sequences) and trigrams (3-word sequences)
    - Capture local context and phrase patterns
    - Used in language modeling, spam detection

### Common Interview Questions:
- How does stemming differ from lemmatization?
- What is TF-IDF and why use it over Bag of Words?
- Explain the role of stopword removal in NLP.
- How would you approach a sentiment classification task?
- What's the difference between POS tagging and NER?
- Why is dependency parsing important for understanding sentences?

---

<small><em>© 2026 AI Nirvana · More Info: https://medium.com/@snigam/a-simple-structured-way-to-prepare-for-ai-ml-interviews-68b2e5830195 · Disclaimer: Provided as is. No liability assumed.</em></small>